# Variables and Parameters

> Terms are thin wrappers around sympy symbols

In [ ]:
#| default_exp term

In [ ]:
#| export
import sympy as sp
import numpy as np
import pandas as pd
from sympy.printing.latex import translate

In [ ]:
#| export

class Term(sp.Symbol):
  "A symbolic term in an SFC model."
  default_value = np.nan

  def __new__(cls, name, description=None, initial_value=None, lag=0, unlagged=None, *args, **kwargs):
    "Create a term."
    symbol_name = f"{name}" if lag == 0 else f"{name}_{{-{-lag}}}"
    obj = super().__new__(cls, symbol_name)
    obj.base_name = name
    if unlagged is not None: obj.unlagged = unlagged
    obj.description = description or (unlagged.description if unlagged is not None else "")
    obj.initial_value = unlagged.initial_value if unlagged is not None else initial_value if initial_value is not None else obj.__class__.default_value
    obj.lag = lag
    return obj


  def __call__(self, lag):
    if isinstance(lag, (pd.DataFrame, pd.Series)):  # needed because pandas calls callable keys
        return self
    if not isinstance(lag, (int, np.integer)) or lag > 0:
        raise ValueError(f"lag must be a non-positive int, got {lag!r}")
    base = self.unlagged if self.lag else self
    total = self.lag + int(lag)
    return base if total == 0 else type(self)(self.base_name, lag=total, unlagged=base)


  def _latex(self, printer):
    # add lag to any subscript
    name, *subs = self.base_name.split("_")
    subs = [translate(s) for s in subs] + ([str(self.lag)] if self.lag else [])
    out = translate(name)
    return out + ("_{" + ",".join(subs) + "}" if subs else "")


In [ ]:
#| hide
t = Term("t")
assert isinstance(t,sp.Symbol)

`Term`s are thin wrappers around Sympy `symbol`s, mainly to implement lags and equations. There are two kinds, `Parameter`s and `Variable`s

In [ ]:
#| export

class Parameter(Term):
  'An exogenous value, not calculated by the solver.'
    
  pass


Declare a parameter with its LaTeX name, and an optional description and initial value. Be sure to store it in a variable for later use in an equation.

In [ ]:
alpha1 = Parameter("alpha_1","Marginal propensity to consume",0.5)

alpha1

alpha_1

In [ ]:
#| hide
assert alpha1.name == "alpha_1"
assert alpha1.description == "Marginal propensity to consume"
assert alpha1.initial_value == 0.5
assert alpha1.lag == 0

In [ ]:
#| export

class Variable(Term):
  "An endogenous value, solved each timestep."
    
  def __lshift__(self, other):
    return sp.Eq(self, other)

  def _numpycode(self, printer, **kwargs):
    return printer._print_Symbol(self)
    

The interface for a Variable is the same as for a `Parameter`, defined with a LaTeX name and optional description and initial value:

In [ ]:
Y = Variable("Y","GDP",100)

Y

Y

In [ ]:
#| hide
assert Y.name == "Y"
assert Y.description == "GDP"
assert Y.initial_value == 100

Instead of setting default values separately for every term, you can alternatively set them for the whole class, such as:

In [ ]:
Parameter.default_value = 0.3
alpha2 = Parameter("alpha_2")

Variable.default_value = 10
G = Variable("G")

In [ ]:
#| hide
assert alpha2.initial_value == 0.3
assert G.initial_value == 10

Or set them for both `Parameters` and `Variables` by using `Term`:

In [ ]:
# Delete the specific defaults if you assigned them, otherwise skip this part
del Parameter.default_value
del Variable.default_value

Term.default_value = 20
gamma = Parameter("gamma")
C = Variable("C")

In [ ]:
#| hide
assert gamma.initial_value == 20
assert C.initial_value == 20

You can add a lag to a term by putting it in parentheses:

In [ ]:
Y(-1)

Y_{-1}

Then use these in equations like any other symbol.

In [ ]:
#| hide
original = Y
lag = Y(-1)

assert lag.unlagged == original
assert lag.base_name == original.name
assert lag.description == original.description
assert lag.lag == -1

assert original(0) == original
assert original(-2) == lag(-1)
assert lag(-1).unlagged == original

Create equations using `<<` in place of the equals sign - this is just a shorthand for creating `Sympy` equations:

In [ ]:
C = Variable("C","Consumption")
I = Variable("I","Investment")
Y = Variable("Y","GDP")

Y << C + I

Eq(Y, C + I)

In [ ]:
#| hide
assert isinstance(Y << C + I, sp.Eq) 
assert (Y << C + I) == sp.Eq(Y,C+I)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()